# 03 章 / 第 1 节 · lm-evaluation-harness 跑英文 benchmark baseline

## 学习目标
- 装并跑通 `lm_eval` CLI(EleutherAI 出品,业界事实标准评测框架)
- 在 Qwen2.5-1.5B-Instruct 上跑 ARC-Easy / HellaSwag / TruthfulQA / MMLU 子集 baseline
- 理解 `acc` vs `acc_norm` 的差异(letter bias 与长度归一化)
- 出英文 baseline 分数表,**后续 SFT / 对齐 / 量化每章用同一脚本对比**

## 对应八股
`liguodongiot/llm-action`:**`llm-interview/llm-eval.md`**

## ⚠️ 运行要求
- Linux / WSL2 + CUDA(`vllm` backend 才快;CPU 也能跑但 100x 慢)
- 完整 MMLU(57 tasks)约 2-4h on 4070;本节只跑 4 task 子集 + MMLU 3 个 subject ≈ 30 min


## 1. 概念背景

### 1.1 为什么用 lm-eval-harness 而不是自己写?

- **可复现**:全社区用同一份 prompt / few-shot / score 计算,你的 70% 和别人的 70% 真能比
- **任务多**:200+ task,从 MMLU/HellaSwag 到 GSM8K/HumanEval 全有
- **多 backend**:`hf`(transformers)/ `vllm`(快 5-10x)/ `openai`(给 API 评)

### 1.2 `acc` vs `acc_norm` 的坑

多选题任务(MMLU/HellaSwag)有两种打分:
- **`acc`** = 模型选哪个 option likelihood 最高
- **`acc_norm`** = likelihood 按 option **token 长度归一化**(长答案天然 likelihood 低,要除掉这个 bias)

**Qwen / Llama 系都有 letter bias**(偏好选 A 或 D),`acc_norm` 抑制这个倾向。
看分数时**优先看 `acc_norm`**;只有 `acc` 时小心 calibration。

### 1.3 Contamination 问题

很多 base 模型训练时已经吃过 MMLU(或其 train 集),分数虚高。SOLAR 论文专门有 contamination 章节。
本节只看**相对差异**(SFT/对齐前后),绝对分数当参考。


## 2. 环境检查


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from scripts.env_check import check
check(extras=("lm_eval", "vllm"))


## 3. 装 lm-eval(如果还没装)


In [ ]:
# 装最新版(支持 Qwen2.5)
# !pip install -q "lm-eval[vllm]" 2>&1 | tail -1
# Linux + CUDA 用 vllm backend 快 5-10x;Windows / Mac 退回 hf backend(慢但能跑)
import importlib
if importlib.util.find_spec("lm_eval") is None:
    print("请先执行:pip install lm-eval[vllm]")
else:
    import lm_eval
    print(f"lm_eval {lm_eval.__version__} 已装")


## 4. 跑评测(Python API 方式)

`lm_eval.simple_evaluate` 是底层 API,比 CLI 更方便嵌进 notebook。

选了 4 个常见任务做 baseline:
| Task | 类型 | 衡量 |
|---|---|---|
| `arc_easy` | 多选 | 小学常识 |
| `hellaswag` | 多选 | 常识推理 |
| `truthfulqa_mc1` | 多选 | 抗 hallucination |
| `mmlu_high_school_*`(3 科) | 多选 | 学术知识 |


In [ ]:
import lm_eval
from lm_eval.models.huggingface import HFLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# backend 选择:有 vllm 用 vllm,没 vllm 退回 transformers
try:
    from lm_eval.models.vllm_causallms import VLLM
    model = VLLM(pretrained=MODEL_NAME, dtype="bfloat16", max_model_len=2048,
                 gpu_memory_utilization=0.8, batch_size="auto")
    print("使用 vLLM backend(快)")
except Exception as e:
    print(f"vLLM 不可用({e}),退回 transformers backend")
    model = HFLM(pretrained=MODEL_NAME, dtype="bfloat16", batch_size=4)


In [ ]:
TASKS = [
    "arc_easy",
    "hellaswag",
    "truthfulqa_mc1",
    "mmlu_high_school_mathematics",
    "mmlu_high_school_computer_science",
    "mmlu_high_school_world_history",
]

# 抽样跑:每个 task 取 100 题(教学跑;真评测把 limit 去掉)
results = lm_eval.simple_evaluate(
    model = model,
    tasks = TASKS,
    num_fewshot = 0,          # 0-shot,instruct 模型不需要 few-shot
    limit = 100,              # 教学跑用 100 题;真评测 = None
    batch_size = "auto",
    log_samples = False,
)

print("\n=== 完成 ===")
print(f"评测的 task 数: {len(results['results'])}")


## 5. 整理 baseline 表


In [ ]:
import pandas as pd

rows = []
for task, scores in results["results"].items():
    rows.append({
        "task": task,
        "acc":      f"{scores.get('acc,none', float('nan'))*100:.1f}%",
        "acc_norm": f"{scores.get('acc_norm,none', float('nan'))*100:.1f}%",
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))

# 存到 CSV 供后续章节对照
import json
pathlib.Path("../checkpoints/03_eval").mkdir(parents=True, exist_ok=True)
df.to_csv("../checkpoints/03_eval/baseline_en.csv", index=False)
with open("../checkpoints/03_eval/baseline_en.json", "w") as f:
    json.dump(results["results"], f, ensure_ascii=False, indent=2, default=str)
print("\n已存到 ../checkpoints/03_eval/baseline_en.{csv,json}")


## 6. 踩坑记录

- **`num_fewshot=0` for instruct,`num_fewshot=5` for base**:base 模型不会 zero-shot 解题,要 5-shot 演示;instruct 反过来,few-shot 反而把它带偏
- **MMLU 不要跑全 57 task**:总样本 ~14k,vLLM 也要 2h+。挑 3-5 个有代表性的 subject 看趋势就够
- **lm-eval 0.4.5+ 字段名变了**:旧版 `acc` 现在叫 `acc,none`(逗号 + none = 默认 filter),代码里要带逗号
- **vLLM `gpu_memory_utilization` 默认 0.9 容易 OOM**:12GB 卡跑 1.5B 配 0.8 才稳
- **truthfulqa_mc1 vs mc2 差异巨大**:mc1 = 找到最对的那个,mc2 = 综合 likelihood;面试更常聊 mc2

## 7. 延伸阅读

- [lm-evaluation-harness 仓库](https://github.com/EleutherAI/lm-evaluation-harness)
- [MMLU 论文 `/paper fetch 2009.03300`]
- [HELM(Stanford 评测,更全)](https://crfm.stanford.edu/helm/)
- 本仓:[`03_eval/02_cmmlu_ceval_zh.ipynb`](02_cmmlu_ceval_zh.ipynb)(中文 benchmark)
- [[Slipbox/lm-eval-task-design]] / [[Slipbox/contamination-and-leakage]]
